![image.png](https://i.imgur.com/a3uAqnb.png)

# **🚀 Object Detection with Faster R-CNN**
In this lab, we will:

✅ Build a **custom Dataset class** for **Pascal VOC dataset**  
✅ Use a **pretrained Faster R-CNN model** for object detection  
✅ Train and evaluate the model  

---

## **1️⃣ Dataset Class**
We use the **Pascal VOC 2007 dataset**, which contains images with **bounding boxes and labels**.  
PyTorch provides a built-in dataset loader: `torchvision.datasets.VOCDetection`.



> **Don't worry about this, just downloading the dataset** 👇



In [ ]:
import os
from IPython.display import clear_output

DATA_DIR = "/content/VOCdata"
os.makedirs(DATA_DIR, exist_ok=True)

BASE = "https://data.brainchip.com/dataset-mirror/voc"

!wget -c "{BASE}/VOCtrainval_06-Nov-2007.tar" -O "{DATA_DIR}/VOCtrainval_06-Nov-2007.tar"
!wget -c "{BASE}/VOCtest_06-Nov-2007.tar"     -O "{DATA_DIR}/VOCtest_06-Nov-2007.tar"

!file "{DATA_DIR}/VOCtrainval_06-Nov-2007.tar"
!tar -tf "{DATA_DIR}/VOCtrainval_06-Nov-2007.tar" | head -n 5

!tar -xf "{DATA_DIR}/VOCtrainval_06-Nov-2007.tar" -C "{DATA_DIR}"
!tar -xf "{DATA_DIR}/VOCtest_06-Nov-2007.tar"     -C "{DATA_DIR}"

clear_output()

!ls -lah "{DATA_DIR}/VOCdevkit/VOC2007" | head
print("✅ VOC 2007 extracted correctly")


total 592K
drwxrwxrwx 7 root root 4.0K Nov  6  2007 .
drwxrwxrwx 3 root root 4.0K Nov  6  2007 ..
drwxrwxrwx 2 root root 264K Nov  6  2007 Annotations
drwxrwxrwx 5 root root 4.0K Nov  6  2007 ImageSets
drwxrwxrwx 2 root root 268K Nov  6  2007 JPEGImages
drwxrwxrwx 2 root root  20K Nov  6  2007 SegmentationClass
drwxrwxrwx 2 root root  20K Nov  6  2007 SegmentationObject
✅ VOC 2007 extracted correctly


In [ ]:
import os
import torchvision
from torchvision.datasets import VOCDetection
from torch.utils.data import DataLoader

### **🔹 Load & Transform Dataset**
data_path = "/content/VOCdata"
transform = torchvision.transforms.Compose([torchvision.transforms.ToTensor()])

# Load Pascal VOC dataset (train & test)
train_dataset = VOCDetection(root=data_path, year="2007", image_set="train", download=False, transform=transform)
test_dataset = VOCDetection(root=data_path, year="2007", image_set="test",download=False, transform=transform)

# Custom collate function to handle variable number of bounding boxes
def custom_collate_fn(data):
  return tuple(zip(*data))

# Create DataLoaders                 (Tip: Use lower batch size if encountered Out-of-Memory (OOM) error in Training)
train_loader = DataLoader(train_dataset, batch_size=1, shuffle=True, num_workers=2, collate_fn=custom_collate_fn, pin_memory=True)
test_loader = DataLoader(test_dataset, batch_size=1, shuffle=False,num_workers=2, collate_fn=custom_collate_fn, pin_memory=True)

In [ ]:
print("Train size:", len(train_dataset))
print("Test size :", len(test_dataset))

Train size: 2501
Test size : 4952


In [ ]:
# Mappings of label names (found in dataset annotation) to integer IDs (or classes) which we will feed to the model
voc_classes = {
    "aeroplane": 0,
    "bicycle": 1,
    "bird": 2,
    "boat": 3,
    "bottle": 4,
    "bus": 5,
    "car": 6,
    "cat": 7,
    "chair": 8,
    "cow": 9,
    "diningtable": 10,
    "dog": 11,
    "horse": 12,
    "motorbike": 13,
    "person": 14,
    "pottedplant": 15,
    "sheep": 16,
    "sofa": 17,
    "train": 18,
    "tvmonitor": 19,
}

#  Reverse of label to class id mapping. needed because the model predictions will be ids and we need to change it to label to visualize it.
reverse_voc_classes = {v: k for k, v in voc_classes.items()}


### **🔹 Why Do We Need a Custom `collate_fn`?**
Unlike classification datasets, where each image has a **fixed shape and label**, object detection images have **variable numbers of bounding boxes**.  

- The default `collate_fn` (which applies `torch.stack`) **doesn't work**, since bounding box tensors have different shapes.  
- Instead, we **return a tuple** that **keeps individual image-label pairs separate**.

##### Before using custom collate_fn:
```python
data = [
    (image1, dict1),  
    (image2, dict2),
    ...  
]
```
##### After:
```python
images_tuple = (image1, image2, ...)  
targets_tuple = (dict1, dict2, ...)   
```

## **2️⃣ Model Class**
We use a **pretrained Faster R-CNN with a MobileNetV3-Large backbone**.

### **🔹 Modify the Model**
- The default model is trained on **COCO dataset** with **91 classes**.
- We modify the classifier to detect **20 Pascal VOC classes**.


## **3️⃣ Training and Validation Loops**
### **🔹 Training Loop**
- The model takes **images & targets** and computes **losses internally**. No need to define the loss.
- We only need to **backpropagate and update the optimizer**.


### **🔹 Validation Loop**
#### **🔹 How Do We Evaluate Object Detection Models?**
To evaluate object detection models like **Faster R-CNN**, we need to measure **how well the predicted bounding boxes match the ground truth boxes**.

![image.png](https://i.imgur.com/MDFxFMX.png)

---

#### **📌 Intersection over Union (IoU)**
✅ We consider a detection **correct** if the predicted box **overlaps significantly** with the ground truth box.  
✅ This is measured using **Intersection over Union (IoU)**, which calculates the **ratio of overlap** between the two boxes.

$$
IoU = \frac{\text{Area of Overlap}}{\text{Area of Union}}
$$

🚀 **Higher IoU = Better Detection!**  


![image.png](https://i.imgur.com/yNNhjwr.png)

---

#### **📌 What is mAP@0.5:0.95?**
mAP (**mean Average Precision**) is the most commonly used **metric for object detection**.

🔹 **mAP@0.5:0.95** means we compute the **average precision** at **different IoU thresholds** from **0.5 to 0.95**, increasing in steps of **0.05**.

- **IoU ≥ 0.5** → Loose match  
- **IoU ≥ 0.75** → Stricter match  
- **IoU ≥ 0.95** → Extremely strict match  

**mAP@0.5:0.95** takes the **average of all these values**, giving us a single number that represents how well the model performs **across different difficulty levels**.


In [ ]:
from IPython.display import clear_output
!pip install torchmetrics
clear_output()

In [ ]:
from torchmetrics.detection.mean_ap import MeanAveragePrecision

# Initialize metric
metric = MeanAveragePrecision(iou_thresholds=[0.5, 0.55, 0.6, 0.65, 0.7, 0.75, 0.8, 0.85, 0.9, 0.95])




## **4️⃣ Running Training & Validation**

## **5️⃣ Visualizing Predictions vs. Ground Truth**


In [ ]:
import random
import torchvision.transforms.functional as F
import matplotlib.pyplot as plt
import matplotlib.patches as patches

In [ ]:
# Select 3 random test images
test_indices = [2777, 4742, 777]

# Set model to evaluation mode
model.eval()




NameError: name 'model' is not defined

### Contributed by: Mohamed Eltayeb & Farah Alshiha

![image.png](https://i.imgur.com/a3uAqnb.png)